# INN reconciliation: Excel vs Lake (agr_id, SA, trx-perimeter)

Checks INN set differences between Excel and Lake for selected period with agreed filters:
- SA active contracts;
- `agr_id is not null`;
- transaction perimeter: `c_trx_class='SA'`, `c_trx_type='S01'`, `cf_trx_stat<>'R'`, `c_nter is not null`, `ods_deleted_flg<>'1'`;
- FIID condition: `c_fiid_acq_grp='RSHB'`.

Includes robust INN normalization with leading zero recovery.

In [ ]:
import re
import pandas as pd
from decimal import Decimal, InvalidOperation
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
report_month = '2026-02-01'
month_start = pd.to_datetime(report_month).strftime('%Y-%m-%d')
month_end = (pd.to_datetime(report_month) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_col_inn = 'ИНН'

sa_type = 'SA'
mem_limit = '10g'

print('report_month =', report_month)
print('month_start =', month_start)
print('month_end =', month_end)
print('excel_path =', excel_path)
print('sa_type =', sa_type)
print('mem_limit =', mem_limit)

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    if not s:
        return None

    # Leading zero recovery for typical broken INN lengths from Excel.
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s

    return s

def to_sql_in_list(values):
    out = []
    for v in values:
        s = str(v).replace("'", "''")
        out.append("'" + s + "'")
    return ', '.join(out)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_base24_fiids')

print('Impala initialized')

## 1) Excel INN set (normalized)

In [ ]:
excel_raw = pd.read_excel(excel_path)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_col_inn not in excel_raw.columns:
    raise ValueError(f'Missing Excel INN column: {excel_col_inn}')

excel_inn_df = excel_raw[[excel_col_inn]].copy()
excel_inn_df.columns = ['inn_raw']
excel_inn_df['inn'] = excel_inn_df['inn_raw'].apply(normalize_numeric_str)
excel_inn_df = excel_inn_df[(excel_inn_df['inn'].notna()) & (excel_inn_df['inn'] != '')].copy()

excel_inn_set_df = excel_inn_df[['inn']].drop_duplicates().sort_values('inn').reset_index(drop=True)

print('excel_inn_cnt =', len(excel_inn_set_df))
display(excel_inn_set_df.head(30))

## 2) Lake INN set with agreed filters (`agr_id` + SA + trx perimeter + FIID=RSHB)

In [ ]:
sql_lake_inn = f"""
with agr as (
  select
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = '{sa_type}'
    and a.abs_agr_id is not null
    and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
),
trx_base as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(ta.n_agr as string) as n_agr
  from ods_alpha.scd1_trx t
  join ods_alpha.scd1_trx_acq ta on cast(ta.n_trx as string) = cast(t.n_trx as string)
  left join ods_alpha.scd1_base24_fiids fa on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
  where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and t.c_nter is not null
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
)
select distinct a.inn
from agr a
join trx_base t on t.n_agr = a.n_agr
where a.inn is not null and a.inn <> ''
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    lake_inn_set_df = imp.fetch(sql_lake_inn)

lake_inn_set_df['inn'] = lake_inn_set_df['inn'].apply(normalize_numeric_str)
lake_inn_set_df = lake_inn_set_df[(lake_inn_set_df['inn'].notna()) & (lake_inn_set_df['inn'] != '')].drop_duplicates().sort_values('inn').reset_index(drop=True)

print('lake_inn_cnt =', len(lake_inn_set_df))
display(lake_inn_set_df.head(30))

## 3) Reconciliation and requested counts

In [ ]:
excel_set = set(excel_inn_set_df['inn'].tolist())
lake_set = set(lake_inn_set_df['inn'].tolist())

both = sorted(excel_set & lake_set)
lake_only = sorted(lake_set - excel_set)
excel_only = sorted(excel_set - lake_set)

recon_kpi = pd.DataFrame([
    {
        'metric': 'inn_with_agr_id_in_both_lake_and_excel',
        'value': len(both)
    },
    {
        'metric': 'inn_with_agr_id_in_lake_but_not_in_excel',
        'value': len(lake_only)
    },
    {
        'metric': 'inn_only_in_excel_not_in_lake',
        'value': len(excel_only)
    },
    {
        'metric': 'lake_total_unique_inn_after_filters',
        'value': len(lake_set)
    },
    {
        'metric': 'excel_total_unique_inn_after_normalization',
        'value': len(excel_set)
    }
])

display(recon_kpi)

both_df = pd.DataFrame({'inn': both})
lake_only_df = pd.DataFrame({'inn': lake_only})
excel_only_df = pd.DataFrame({'inn': excel_only})

print('both sample')
display(both_df.head(30))
print('lake_only sample')
display(lake_only_df.head(30))
print('excel_only sample')
display(excel_only_df.head(30))

## 4) QC for INN normalization (leading zero losses and parse issues)

In [ ]:
excel_qc = excel_raw[[excel_col_inn]].copy()
excel_qc.columns = ['inn_raw']
excel_qc['inn_raw_str'] = excel_qc['inn_raw'].astype(str)
excel_qc['inn_norm'] = excel_qc['inn_raw'].apply(normalize_numeric_str)
excel_qc['raw_digits'] = excel_qc['inn_raw_str'].str.replace(r'\D', '', regex=True)
excel_qc['raw_len'] = excel_qc['raw_digits'].str.len()
excel_qc['norm_len'] = excel_qc['inn_norm'].fillna('').str.len()
excel_qc['leading_zero_restored'] = excel_qc['raw_len'].isin([9, 11]) & excel_qc['norm_len'].isin([10, 12])
excel_qc['parse_failed'] = excel_qc['inn_norm'].isna()

qc_kpi = pd.DataFrame([
    {'metric': 'excel_rows_total', 'value': int(len(excel_qc))},
    {'metric': 'excel_rows_parse_failed', 'value': int(excel_qc['parse_failed'].sum())},
    {'metric': 'excel_rows_leading_zero_restored', 'value': int(excel_qc['leading_zero_restored'].sum())},
    {'metric': 'excel_unique_inn_after_normalization', 'value': int(excel_qc['inn_norm'].dropna().nunique())}
])

display(qc_kpi)

display(
    excel_qc.loc[excel_qc['leading_zero_restored'], ['inn_raw', 'inn_raw_str', 'raw_digits', 'inn_norm']]
    .drop_duplicates()
    .head(30)
)

display(
    excel_qc.loc[excel_qc['parse_failed'], ['inn_raw', 'inn_raw_str']]
    .head(30)
)